In [18]:
from typing import List, Tuple

def build_token_char_spans(tokens: List[str]) -> List[Tuple[int, int]]:
    """Character spans for tokens in the canonical CoNLL text: ' '.join(tokens)."""
    spans: List[Tuple[int, int]] = []
    pos = 0
    for i, tok in enumerate(tokens):
        start = pos
        end = start + len(tok)
        spans.append((start, end))
        pos = end + (1 if i < len(tokens) - 1 else 0)
    return spans

In [ ]:
from typing import Dict, List
import re
VALID_LABELS = {"PER", "LOC", "ORG", "MISC"}
LABEL_PATTERN = '|'.join(VALID_LABELS)
SPAN_RE = re.compile(rf'<SPAN><LABEL>({LABEL_PATTERN})</LABEL>(.*?)</SPAN>', re.DOTALL)

def parse_entities_from_tagged_output(tagged_text: str) -> Dict:
    """
    Parse <SPAN><LABEL>..</LABEL>entity</SPAN> blocks, return entities with char offsets,
    and reconstruct plain text by removing span markup.
    """
    cursor = 0
    plain_parts: List[str] = []
    entities: List[Dict] = []

    for match in SPAN_RE.finditer(tagged_text):
        plain_parts.append(tagged_text[cursor:match.start()])

        label = match.group(1).strip()
        entity_text = match.group(2)
        entity_start = sum(len(p) for p in plain_parts)
        entity_end = entity_start + len(entity_text)

        plain_parts.append(entity_text)
        entities.append({
            "entity": entity_text,
            "label": label,
            "start": entity_start,
            "end": entity_end,
        })
        cursor = match.end()
    plain_parts.append(tagged_text[cursor:])
    reconstructed_text = "".join(plain_parts)

    malformed = (
        "<SPAN>" in reconstructed_text
        or "</SPAN>" in reconstructed_text
        or "<LABEL>" in reconstructed_text
        or "</LABEL>" in reconstructed_text
    )
    invalid_labels = [ent for ent in entities if ent["label"] not in VALID_LABELS]
    
    return {
        "entities": entities,
        "reconstructed_text": reconstructed_text,
        "malformed_markup": malformed,
        "invalid_label_count": len(invalid_labels),
        "span_count": len(entities),
    }

In [40]:
def entities_to_bio_tags(tokens: List[str], entities: List[Dict]) -> List[str]:
    """Convert entity char spans to token-level BIO tags for the same tokenization as input text."""
    token_spans = build_token_char_spans(tokens)
    tags = ["O"] * len(tokens)

    entities_sorted = sorted(entities, key=lambda x: (x["start"], x["end"]))
    for ent in entities_sorted:
        label = ent.get("label")
        if label not in VALID_LABELS:
            continue

        e_start = int(ent.get("start", -1))
        e_end = int(ent.get("end", -1))
        if e_start < 0 or e_end <= e_start:
            continue

        covered = [
            i for i, (t_start, t_end) in enumerate(token_spans)
            if max(t_start, e_start) < min(t_end, e_end)
        ]
        if not covered:
            continue

        tags[covered[0]] = f"B-{label}"
        for idx in covered[1:]:
            tags[idx] = f"I-{label}"

    return tags

In [57]:
input_text = "6. Jeremie Collomb-Patton ( from France ) 23.87"
tokens = input_text.split(" ")
print(tokens)

print(build_token_char_spans(tokens))

['6.', 'Jeremie', 'Collomb-Patton', '(', 'from', 'France', ')', '23.87']
[(0, 2), (3, 10), (11, 25), (26, 27), (28, 32), (33, 39), (40, 41), (42, 47)]


In [60]:
tagged_text = "<SPAN><LABEL>PER</LABEL>Jeremie Collomb-Patton</SPAN> ( from <SPAN><LABEL>LOC</LABEL>France</SPAN> ) 23.87"

info = parse_entities_from_tagged_output(tagged_text)
print(info)

['', 'Jeremie Collomb-Patton', ' ( from ', 'France', ' ) 23.87']
{'entities': [{'entity': 'Jeremie Collomb-Patton', 'label': 'PER', 'start': 0, 'end': 22}, {'entity': 'France', 'label': 'LOC', 'start': 30, 'end': 36}], 'reconstructed_text': 'Jeremie Collomb-Patton ( from France ) 23.87', 'malformed_markup': False, 'invalid_label_count': 0, 'span_count': 2}


In [ ]:
tags = entities_to_bio_tags(tokens, info["entities"])
for token, tag in zip(tokens, tags):
    print(f"{token}\t{tag}")

NOISE	B-PER
NOISE	I-PER
Jeremie	I-PER
Collomb-Patton	B-LOC
(	I-LOC
from	O
France	O
)	O
23.87	O
